# Naive Bayes Classifier

In [ ]:
import numpy as npimport pandas as pdfrom sklearn.model_selection import train_test_splitfrom sklearn.datasets import make_classification

In [ ]:
df = pd.read_csv("datasets/airlines_delay.csv", sep=",")AirlineUnique = df.Airline.unique()AirportFromUnique = df.AirportFrom.unique()AirportToUnique = df.AirportTo.unique()Airlinelst = list(range(len(AirlineUnique)))df['NumAirline'] = df['Airline']df['NumAirline'].replace(AirlineUnique, Airlinelst, inplace=True)AirportFromlst = list(range(len(AirportFromUnique)))df['NumAirportFrom'] = df['AirportFrom']df['NumAirportFrom'].replace(AirportFromUnique, AirportFromlst, inplace=True)AirportTolst = list(range(len(AirportToUnique)))df['NumAirportTo'] = df['AirportTo']df['NumAirportTo'].replace(AirportToUnique, AirportTolst, inplace=True)df = df.sample(n=10000)X = df[["Length","NumAirline","NumAirportFrom","NumAirportTo","DayOfWeek"]]y = df['Class']

In [ ]:
X_simulated_small, y_simulated_small = make_classification(n_samples=300, n_features=6, n_classes=2, random_state=1)X_simulated_large, y_simulated_large = make_classification(n_samples=15000, n_features=6, n_classes=2, random_state=1)

In [ ]:
def max_class(x, prior_list, mean_list, variance_list):    likelihoods=[]    post=[]    for idx in range(2):        likelihood_num = np.exp((-1/2)*((x-mean_list[idx])**2)/(2*variance_list[idx]))        likelihood_den = np.sqrt(2*np.pi*variance_list[idx])        likelihoods.append(likelihood_num/likelihood_den)    post.append(np.log(prior_list[0]) + np.sum(np.log(likelihoods[0])))    post.append(np.log(prior_list[1]) + np.sum(np.log(likelihoods[1])))    return np.argmax(post)from multiprocessing.pool import ThreadPooldef Parallel_NB(X_train, X_test, y_train):    n=len(X_train)    m=X_train.shape[1]    prior_list=np.zeros(2, dtype=float)    mean_list=np.zeros((2,m), dtype=float)    variance_list=np.zeros((2,m), dtype=float)    for index in range(2):        sub_df = X_train[y_train==index]        prior_list[index] = len(sub_df)/n        mean_list[index,:] = sub_df.mean(axis=0)        variance_list[index,:] = sub_df.var(axis=0)    X_testing = X_test.values    Xis=[row.tolist() for row in X_testing]    pool = ThreadPool(5)    y_pred = [pool.apply(max_class, args=(Xi, prior_list, mean_list, variance_list)) for Xi in Xis]    return y_preddef test_accuracy(true, pred):    correct=sum(a==b for a,b in zip(true,pred))    return correct/len(true)

In [ ]:
# Simulated Study Naive Bayes (small data)start=time.time()acc=[]for i in range(10):    X_train,X_test,y_train,y_test=train_test_split(X_simulated_small,y_simulated_small,test_size=0.2,random_state=i)    X_train=pd.DataFrame(X_train)    X_test=pd.DataFrame(X_test)    y_train=pd.Series(y_train)    pred=Parallel_NB(X_train,X_test,y_train)    acc.append(test_accuracy(y_test,pred))print('Prediction accuracy of model:',sum(acc)/len(acc))print('Training time for Naive Bayes:',time.time()-start)

In [ ]:
# Simulated Study Naive Bayes (large data)start=time.time()acc=[]for i in range(10):    X_train,X_test,y_train,y_test=train_test_split(X_simulated_large,y_simulated_large,test_size=0.2,random_state=i)    X_train=pd.DataFrame(X_train)    X_test=pd.DataFrame(X_test)    y_train=pd.Series(y_train)    pred=Parallel_NB(X_train,X_test,y_train)    acc.append(test_accuracy(y_test,pred))print('Prediction accuracy of model:',sum(acc)/len(acc))print('Training time for Naive Bayes:',time.time()-start)

In [ ]:
# Real data study for Naive Bayesstart=time.time()acc=[]for i in range(10):    X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=i,shuffle=True)    pred=Parallel_NB(X_train,X_test,y_train)    acc.append(test_accuracy(y_test.values,pred))print('Prediction accuracy of model:',sum(acc)/len(acc))print('Training time for Naive Bayes:',time.time()-start)